In [ ]:
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
from Bio.SeqUtils import seq1
from pathlib import Path
import json

# Based on analysis/dssp_calculation.ipynb: run DSSP on the rBMRB test-split structures and
# write only resnum / residue_type / ss_class.
# The other fields (s2/exposure/plddt/tdi/tdn/p_exchange/ph/temperature) come from other sources
# and are not computed here.
pdb_dir = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
out_dir = Path("../../../data/analysis/Fig4/dssp_results")

# mkdssp must be on PATH; the original run used the conda-forge build (DSSP 4.6.1).
# Make sure `which mkdssp` resolves to that build and not to another installation.
# If mkdssp fails with 'Could not load dictionary mmcif_pdbx.dic', point libcifpp to its data, e.g.
#   export LIBCIFPP_DATA_DIR=$CONDA_PREFIX/share/libcifpp
MKDSSP_BIN = "mkdssp"

def classify_ss(code):
    if code in ("H", "G", "I"): return "helix"
    if code in ("E", "B"):       return "sheet"
    return "coil"

p = PDBParser(QUIET=True)
per_residue_records = {}   # entry -> [{"resnum", "residue_type", "ss_class"}, ...]

for pdb_file in sorted(pdb_dir.glob("*.pdb")):
    entry = pdb_file.stem
    structure = p.get_structure(entry, pdb_file)
    model = structure[0]

    try:
        dssp = DSSP(model, str(pdb_file), file_type="PDB", dssp=MKDSSP_BIN)
    except Exception as e:
        print(f"DSSP failed: {pdb_file.name}: {e}")
        dssp = {}

    records = []
    for chain in model:
        for residue in chain:
            if residue.id[0] != " ":   # skip water / hetero residues, standard amino acids only
                continue
            resnum = residue.id[1]
            try:
                residue_type = seq1(residue.resname)
            except Exception:
                residue_type = "X"

            key = (chain.id, residue.id)
            if key in dssp:
                ss_class = classify_ss(dssp[key][2])
            else:
                ss_class = None   # residues DSSP does not cover (chain ends, missing backbone atoms) get None instead of being dropped

            records.append({
                "resnum": resnum,
                "residue_type": residue_type,
                "ss_class": ss_class,
            })

    per_residue_records[entry] = records

out_json = out_dir / "per_residue_records.json"
with open(out_json, "w") as f:
    json.dump(per_residue_records, f, indent=4)

n_res = sum(len(v) for v in per_residue_records.values())
n_missing_ss = sum(1 for v in per_residue_records.values() for r in v if r["ss_class"] is None)
print(f"{len(per_residue_records)} proteins, {n_res} residues total, "
      f"{n_missing_ss} residues with ss_class=None")
print(f"Saved to {out_json}")